# 03 · Casablanca Taxi Simulation (Gravity Model + OSM Routing)

This notebook generates realistic synthetic taxi trips for Casablanca using:
- Estimated RGPH-2024 population per arrondissement
- Zone bounding boxes and densities
- OSM POI activity and drivable network
- Gravity OD modeling calibrated from a sampled subset of Porto trips
- Parallel route generation and PySpark Parquet output

## ADR · Why Gravity + OSM Routing?

**Context**: simple coordinate transformation does not preserve realistic destination attractiveness or road-constrained movement.

**Decision**: use a gravity model for OD flows and OSM shortest paths for trajectories.

**Consequences**: better realism for demand forecasting and geospatial analytics, at the cost of more route compute (mitigated with parallel routing on a subset).

In [1]:
# %pip install osmnx geopandas shapely folium networkx matplotlib pyarrow pyspark

import json
import math
import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import pytz
from concurrent.futures import ProcessPoolExecutor
from folium.plugins import HeatMap
from shapely.geometry import Polygon

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)
print('Imports OK')

Imports OK


In [2]:
@dataclass
class SimulationConfig:
    city_name: str = 'Casablanca, Morocco'
    zones_csv: str = '../metadata/zone_mapping.csv'
    porto_csv_s3: str = 's3a://taasim/raw/porto-trips/train.csv'
    porto_csv_local: str = './raw/porto-trips/train.csv'
    graphml_cache: str = '/tmp/casablanca_drive.graphml'

    # Profile selector: quick for iteration, full for heavy generation.
    run_profile: str = 'quick'
    total_trips: int = 1_000_000
    detailed_route_trips: int = 10_000

    taxi_pool_size: int = 442
    beta_default: float = 1.8

    # Porto is used only as a sampled proxy for calibration and temporal behavior.
    porto_simulation_fraction: float = 0.08
    porto_simulation_max_rows: int = 300_000
    porto_beta_sample_size: int = 120_000
    porto_time_sample_size: int = 300_000

    output_s3: str = 's3a://taasim/curated/simulated_casa_trips/'
    output_local: str = './data/simulated_casa_trips/'
    workers: int = max(2, (os.cpu_count() or 4) // 2)

CFG = SimulationConfig()

PROFILE_PRESETS = {
    'quick': {
        'total_trips': 200_000,
        'detailed_route_trips': 5_000,
        'porto_simulation_fraction': 0.05,
        'porto_simulation_max_rows': 120_000,
        'porto_beta_sample_size': 60_000,
        'porto_time_sample_size': 120_000,
    },
    'full': {
        'total_trips': 1_000_000,
        'detailed_route_trips': 50_000,
        'porto_simulation_fraction': 0.08,
        'porto_simulation_max_rows': 300_000,
        'porto_beta_sample_size': 120_000,
        'porto_time_sample_size': 300_000,
    },
}

if CFG.run_profile not in PROFILE_PRESETS:
    raise ValueError(f'Unknown profile: {CFG.run_profile}. Choose one of {list(PROFILE_PRESETS)}')

for k, v in PROFILE_PRESETS[CFG.run_profile].items():
    setattr(CFG, k, v)

POPULATION_2024_EST = {
    'Sidi Belyout': 188000, 'Maarif': 260000, 'Anfa': 220000, 'Hay Hassani': 420000,
    'Mers Sultan': 190000, 'Ain Chock': 315000, 'Hay Mohammadi': 245000, 'Sidi Bernoussi': 365000,
    'Ain Sebaa': 210000, 'Roches Noires': 165000, 'Sidi Moumen': 390000, 'El Fida': 185000,
    'Mechouar': 52000, 'Ben Msik': 355000, 'Sbata': 240000, 'Moulay Rachid': 415000
}

spark = (SparkSession.builder
         .appName('TaaSim-Casa-Gravity-Sim')
         .config('spark.sql.adaptive.enabled', 'true')
         .getOrCreate())

print('Spark:', spark.version)
print('Active profile:', CFG.run_profile)
print('total_trips:', CFG.total_trips, '| detailed_route_trips:', CFG.detailed_route_trips)

Spark: 3.5.0
Active profile: quick
total_trips: 200000 | detailed_route_trips: 5000


## 1) Load and Prepare Zone Data

In [3]:
def haversine_km(lon1, lat1, lon2, lat2):
    r = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*r*math.asin(math.sqrt(max(a, 0.0)))

def bbox_area_km2(lon_min, lon_max, lat_min, lat_max):
    mean_lat = (lat_min + lat_max) / 2.0
    km_lon = 111.32 * math.cos(math.radians(mean_lat))
    km_lat = 111.32
    return max(abs(lon_max-lon_min)*km_lon*abs(lat_max-lat_min)*km_lat, 0.01)

def load_zones(cfg):
    zones_path = cfg.zones_csv if Path(cfg.zones_csv).exists() else '/home/chicken/Desktop/DesktopFiles/BigDataAvancee/project/TaaSim-casablanca/metadata/zone_mapping.csv'
    z = pd.read_csv(zones_path).rename(columns={'arrondissement_id': 'zone_id'})
    z['population'] = z['zone_name'].map(POPULATION_2024_EST).fillna(np.median(list(POPULATION_2024_EST.values()))).astype(int)

    polys, areas, clons, clats = [], [], [], []
    for r in z.itertuples(index=False):
        polys.append(Polygon([(r.lon_min, r.lat_min), (r.lon_max, r.lat_min), (r.lon_max, r.lat_max), (r.lon_min, r.lat_max)]))
        areas.append(bbox_area_km2(r.lon_min, r.lon_max, r.lat_min, r.lat_max))
        clons.append((r.lon_min + r.lon_max) / 2)
        clats.append((r.lat_min + r.lat_max) / 2)

    gdf = gpd.GeoDataFrame(z, geometry=polys, crs='EPSG:4326')
    gdf['area_km2'] = areas
    gdf['density'] = gdf['population'] / gdf['area_km2']
    gdf['centroid_lon'] = clons
    gdf['centroid_lat'] = clats
    return gdf

zones_gdf = load_zones(CFG)
zones_gdf[['zone_id', 'zone_name', 'population', 'area_km2', 'density']].head()

,zone_id,zone_name,population,area_km2,density
0,1,Sidi Belyout,188000,9.288972,20239.053097
1,2,Maarif,260000,13.937497,18654.712656
2,3,Anfa,220000,23.221757,9473.874029
3,4,Hay Hassani,420000,34.078317,12324.552460
4,5,Mers Sultan,190000,10.841217,17525.707124


In [4]:
def load_graph(cfg):
    if Path(cfg.graphml_cache).exists():
        return ox.load_graphml(cfg.graphml_cache)
    G = ox.graph_from_place(cfg.city_name, network_type='drive')
    ox.save_graphml(G, cfg.graphml_cache)
    return G

def load_pois(city_name):
    tags = {'shop': True, 'office': True, 'public_transport': ['platform', 'station', 'stop_position'], 'amenity': ['bus_station', 'taxi']}
    pois = ox.features_from_place(city_name, tags=tags).reset_index(drop=True)
    return gpd.GeoDataFrame(pois, geometry='geometry', crs='EPSG:4326')

def enrich_activity(zones, pois):
    pts = pois[pois.geometry.geom_type.isin(['Point', 'MultiPoint'])][['geometry']].copy()
    if pts.empty:
        zones['poi_count'] = 0
    else:
        joined = gpd.sjoin(pts, zones[['zone_id', 'geometry']], predicate='within', how='left')
        cnt = joined.groupby('zone_id').size().rename('poi_count')
        zones = zones.merge(cnt, on='zone_id', how='left')
        zones['poi_count'] = zones['poi_count'].fillna(0).astype(int)
    zones['employment_proxy'] = 0.3 * zones['population']
    zones['attractiveness'] = zones['employment_proxy'] + 0.7 * zones['poi_count']
    return zones

G_casa = load_graph(CFG)
pois_gdf = load_pois(CFG.city_name)
zones_gdf = enrich_activity(zones_gdf, pois_gdf)
zones_gdf[['zone_id', 'zone_name', 'poi_count', 'attractiveness']].head()

,zone_id,zone_name,poi_count,attractiveness
0,1,Sidi Belyout,208,56545.6
1,2,Maarif,752,78526.4
2,3,Anfa,20,66014.0
3,4,Hay Hassani,305,126213.5
4,5,Mers Sultan,390,57273.0


## 2) Calibrate Beta from Porto

In [5]:
def read_porto_df(spark, cfg):
    paths = [cfg.porto_csv_s3, cfg.porto_csv_local, '/home/chicken/Desktop/DesktopFiles/BigDataAvancee/project/TaaSim-casablanca/raw/porto-trips/train.csv']
    last = None
    for p in paths:
        try:
            df = (spark.read.option('header', True).csv(p)
                  .select('TIMESTAMP', 'POLYLINE', 'CALL_TYPE', 'DAY_TYPE')
                  .filter(F.col('POLYLINE').isNotNull())
                  .filter(F.col('POLYLINE') != '[]')
                  .cache())
            n_raw = df.count()
            frac = min(1.0, cfg.porto_simulation_fraction, cfg.porto_simulation_max_rows / max(n_raw, 1))
            sim_df = df.sample(False, frac, seed=42).cache()
            n_sim = sim_df.count()
            print('Porto source:', p)
            print(f'Using sampled Porto subset for simulation: {n_sim:,}/{n_raw:,} rows (fraction={frac:.4f})')
            return sim_df
        except Exception as e:
            last = e
            print('Failed:', p, e)
    raise RuntimeError(f'Unable to read Porto dataset: {last}')

def polyline_distance_km(polyline_str):
    try:
        coords = json.loads(polyline_str)
        if not isinstance(coords, list) or len(coords) < 2:
            return None
        total = 0.0
        prev = coords[0]
        for cur in coords[1:]:
            total += haversine_km(prev[0], prev[1], cur[0], cur[1])
            prev = cur
        return total if total > 0 else None
    except Exception:
        return None

def estimate_beta(porto_pdf, fallback=1.8):
    d = porto_pdf['distance_km'].dropna()
    d = d[(d > 0.2) & (d < 80)]
    if len(d) < 2000:
        return fallback, d
    bins = np.logspace(np.log10(0.2), np.log10(50), 25)
    hist, edges = np.histogram(d, bins=bins, density=True)
    mids = np.sqrt(edges[:-1] * edges[1:])
    m = hist > 0
    if m.sum() < 3:
        return fallback, d
    slope, _ = np.polyfit(np.log(mids[m]), np.log(hist[m]), 1)
    beta = float(np.clip(-slope, 1.2, 2.5))
    return beta, d

porto_sdf = read_porto_df(spark, CFG)
frac = min(1.0, CFG.porto_beta_sample_size / max(porto_sdf.count(), 1))
porto_pdf = porto_sdf.select('POLYLINE').sample(False, frac, seed=42).toPandas()
porto_pdf['distance_km'] = porto_pdf['POLYLINE'].map(polyline_distance_km)
beta_hat, porto_distances = estimate_beta(porto_pdf, CFG.beta_default)
print('Estimated beta:', beta_hat)

Porto source: s3a://taasim/raw/porto-trips/train.csv
Using sampled Porto subset for simulation: 85,428/1,704,769 rows (fraction=0.0500)
Estimated beta: 1.2


## 3) Gravity Matrix, 4) OD Generation, 5) Parallel Routing

In [ ]:
GLOBAL_GRAPH = None

def build_gravity(zones, beta):
    z = zones.sort_values('zone_id').reset_index(drop=True)
    n = len(z)
    pop = z['population'].to_numpy(float)
    att = z['attractiveness'].to_numpy(float)
    lon = z['centroid_lon'].to_numpy(float)
    lat = z['centroid_lat'].to_numpy(float)

    dist = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            if i == j:
                dist[i, j] = max(math.sqrt(float(z.iloc[i]['area_km2']) / math.pi), 0.5)
            else:
                dist[i, j] = max(haversine_km(lon[i], lat[i], lon[j], lat[j]), 0.5)

    tij = np.zeros((n, n), dtype=float)
    for i in range(n):
        tij[i, :] = (pop[i] * att) / np.power(dist[i, :], beta)

    row_sum = tij.sum(axis=1, keepdims=True)
    probs = np.divide(tij, row_sum, out=np.zeros_like(tij), where=row_sum > 0)
    return z, probs

def sample_od(z, probs, n_trips):
    zone_ids = z['zone_id'].to_numpy()
    p_o = z['population'].to_numpy(float)
    p_o = p_o / p_o.sum()

    origins = np.random.choice(zone_ids, size=n_trips, p=p_o)
    idx = {zid: i for i, zid in enumerate(zone_ids)}
    dest = np.empty(n_trips, dtype=int)

    for zid in zone_ids:
        m = origins == zid
        k = int(m.sum())
        if k == 0:
            continue
        dest[m] = np.random.choice(zone_ids, size=k, p=probs[idx[zid]])

    od = pd.DataFrame({
        'trip_id': np.arange(1, n_trips + 1),
        'origin_zone': origins,
        'destination_zone': dest
    })
    return od

def haversine_km_vec(lon1, lat1, lon2, lat2):
    r = 6371.0088
    lon1 = np.radians(np.asarray(lon1, dtype=float))
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lon2 = np.radians(np.asarray(lon2, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * (np.sin(dlon / 2.0) ** 2)
    return 2.0 * r * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

def _init_worker(graphml):
    global GLOBAL_GRAPH
    GLOBAL_GRAPH = ox.load_graphml(graphml)

def _route_task(pair):
    global GLOBAL_GRAPH
    o_node, d_node = pair
    try:
        G = GLOBAL_GRAPH
        path = nx.shortest_path(G, int(o_node), int(d_node), weight='length')
        coords = [[float(G.nodes[n]['x']), float(G.nodes[n]['y'])] for n in path]

        km = 0.0
        for u, v in zip(path[:-1], path[1:]):
            ed = G.get_edge_data(u, v)
            if ed:
                km += min(e.get('length', 0.0) for e in ed.values()) / 1000.0

        return int(o_node), int(d_node), coords, km
    except Exception:
        return int(o_node), int(d_node), None, None

def attach_coords_and_routes(od, zones, cfg, G):
    out = od.copy()

    zone_bounds = zones[['zone_id', 'lon_min', 'lon_max', 'lat_min', 'lat_max']].copy()
    out = out.merge(
        zone_bounds.rename(columns={
            'zone_id': 'origin_zone',
            'lon_min': 'o_lon_min', 'lon_max': 'o_lon_max',
            'lat_min': 'o_lat_min', 'lat_max': 'o_lat_max'
        }),
        on='origin_zone', how='left'
    )
    out = out.merge(
        zone_bounds.rename(columns={
            'zone_id': 'destination_zone',
            'lon_min': 'd_lon_min', 'lon_max': 'd_lon_max',
            'lat_min': 'd_lat_min', 'lat_max': 'd_lat_max'
        }),
        on='destination_zone', how='left'
    )

    # Vectorized coordinate sampling inside each zone bbox.
    out['origin_lon'] = np.random.uniform(out['o_lon_min'].to_numpy(), out['o_lon_max'].to_numpy())
    out['origin_lat'] = np.random.uniform(out['o_lat_min'].to_numpy(), out['o_lat_max'].to_numpy())
    out['destination_lon'] = np.random.uniform(out['d_lon_min'].to_numpy(), out['d_lon_max'].to_numpy())
    out['destination_lat'] = np.random.uniform(out['d_lat_min'].to_numpy(), out['d_lat_max'].to_numpy())

    out = out.drop(columns=['o_lon_min', 'o_lon_max', 'o_lat_min', 'o_lat_max', 'd_lon_min', 'd_lon_max', 'd_lat_min', 'd_lat_max'])

    if not Path(cfg.graphml_cache).exists():
        ox.save_graphml(G, cfg.graphml_cache)

    detailed = out.sample(n=min(cfg.detailed_route_trips, len(out)), random_state=42).copy()

    # Vectorized nearest-node lookup is much faster than per-trip calls.
    o_nodes = ox.nearest_nodes(
        G,
        X=detailed['origin_lon'].to_numpy(),
        Y=detailed['origin_lat'].to_numpy()
    )
    d_nodes = ox.nearest_nodes(
        G,
        X=detailed['destination_lon'].to_numpy(),
        Y=detailed['destination_lat'].to_numpy()
    )

    detailed['o_node'] = np.asarray(o_nodes, dtype=np.int64)
    detailed['d_node'] = np.asarray(d_nodes, dtype=np.int64)

    pair_df = detailed[['o_node', 'd_node']].drop_duplicates().copy()
    tasks = [tuple(x) for x in pair_df[['o_node', 'd_node']].to_numpy()]
    routed = {}

    with ProcessPoolExecutor(max_workers=cfg.workers, initializer=_init_worker, initargs=(cfg.graphml_cache,)) as ex:
        for o_node, d_node, pl, km in ex.map(_route_task, tasks, chunksize=200):
            routed[(int(o_node), int(d_node))] = (pl, km)

    key_series = list(zip(detailed['o_node'].astype(int), detailed['d_node'].astype(int)))
    detailed['polyline'] = [routed.get(k, (None, None))[0] for k in key_series]
    detailed['distance_km'] = [routed.get(k, (None, None))[1] for k in key_series]

    bad = detailed['polyline'].isna()
    if bad.any():
        b = detailed.loc[bad, ['origin_lon', 'origin_lat', 'destination_lon', 'destination_lat']]
        detailed.loc[bad, 'polyline'] = [
            [[float(olon), float(olat)], [float(dlon), float(dlat)]]
            for olon, olat, dlon, dlat in zip(
                b['origin_lon'].to_numpy(),
                b['origin_lat'].to_numpy(),
                b['destination_lon'].to_numpy(),
                b['destination_lat'].to_numpy()
            )
        ]
        detailed.loc[bad, 'distance_km'] = haversine_km_vec(
            b['origin_lon'].to_numpy(),
            b['origin_lat'].to_numpy(),
            b['destination_lon'].to_numpy(),
            b['destination_lat'].to_numpy()
        )
    detailed['route_generated'] = True

    rem = out[~out['trip_id'].isin(detailed['trip_id'])].copy()
    cent = zones[['zone_id', 'centroid_lon', 'centroid_lat']]
    rem = rem.merge(
        cent.rename(columns={'zone_id': 'origin_zone', 'centroid_lon': 'o_clon', 'centroid_lat': 'o_clat'}),
        on='origin_zone', how='left'
    )
    rem = rem.merge(
        cent.rename(columns={'zone_id': 'destination_zone', 'centroid_lon': 'd_clon', 'centroid_lat': 'd_clat'}),
        on='destination_zone', how='left'
    )
    rem['polyline'] = [
        [[float(olon), float(olat)], [float(dlon), float(dlat)]]
        for olon, olat, dlon, dlat in zip(
            rem['o_clon'].to_numpy(),
            rem['o_clat'].to_numpy(),
            rem['d_clon'].to_numpy(),
            rem['d_clat'].to_numpy()
        )
    ]
    rem['distance_km'] = haversine_km_vec(
        rem['o_clon'].to_numpy(),
        rem['o_clat'].to_numpy(),
        rem['d_clon'].to_numpy(),
        rem['d_clat'].to_numpy()
    )
    rem['route_generated'] = False

    cols = [
        'trip_id', 'origin_zone', 'destination_zone',
        'origin_lon', 'origin_lat', 'destination_lon', 'destination_lat',
        'polyline', 'distance_km', 'route_generated'
    ]
    return pd.concat([detailed[cols], rem[cols]], ignore_index=True)

z_sorted, od_probs = build_gravity(zones_gdf, beta_hat)
synthetic_od = sample_od(z_sorted, od_probs, CFG.total_trips)
routes_pdf = attach_coords_and_routes(synthetic_od, zones_gdf, CFG, G_casa)
routes_pdf.head()

## 6) Add Temporal Dimension from Porto (hour profile + timezone)

In [ ]:
def temporal_profiles(porto_sdf, cfg):
    frac = min(1.0, cfg.porto_time_sample_size / max(porto_sdf.count(), 1))
    p = porto_sdf.sample(False, frac, seed=42).toPandas()
    p['TIMESTAMP'] = pd.to_numeric(p['TIMESTAMP'], errors='coerce')
    p = p.dropna(subset=['TIMESTAMP'])
    p['TIMESTAMP'] = p['TIMESTAMP'].astype(np.int64)

    lisbon = pytz.timezone('Europe/Lisbon')
    dt_lis = pd.to_datetime(p['TIMESTAMP'], unit='s', utc=True).dt.tz_convert(lisbon)
    p['hour'] = dt_lis.dt.hour
    p['is_weekend'] = dt_lis.dt.dayofweek >= 5

    hp = p['hour'].value_counts(normalize=True).sort_index().reindex(range(24), fill_value=0)
    wk = float(p['is_weekend'].mean())
    cp = p['CALL_TYPE'].fillna('B').value_counts(normalize=True)
    for k in ['A', 'B', 'C']:
        if k not in cp.index:
            cp.loc[k] = 0.0
    cp = cp[['A', 'B', 'C']]
    return hp, wk, cp

def sample_timestamps(n, hour_prob, weekend_prob):
    base = pd.Timestamp('2024-01-01 00:00:00', tz='Africa/Casablanca')
    d_off = np.random.randint(0, 365, size=n)
    hrs = np.random.choice(np.arange(24), size=n, p=hour_prob.values)
    ts_local, day_type = [], []
    for i in range(n):
        dt = base + pd.Timedelta(days=int(d_off[i]), hours=int(hrs[i]), minutes=int(np.random.randint(0, 60)), seconds=int(np.random.randint(0, 60)))
        if np.random.random() < weekend_prob and dt.dayofweek < 5:
            dt = dt + pd.Timedelta(days=int(5-dt.dayofweek))
        elif np.random.random() >= weekend_prob and dt.dayofweek >= 5:
            dt = dt + pd.Timedelta(days=int(7-dt.dayofweek))
        ts_local.append(dt)
        day_type.append('B' if dt.dayofweek >= 5 else 'A')
    ts = pd.DatetimeIndex(ts_local).tz_convert('UTC').view('int64') // 10**9
    return ts.astype(np.int64), np.array(day_type, dtype=object)

hour_prob, weekend_prob, call_prob = temporal_profiles(porto_sdf, CFG)
ts_unix, day_types = sample_timestamps(len(routes_pdf), hour_prob, weekend_prob)
routes_pdf['timestamp'] = ts_unix
routes_pdf['day_type'] = day_types
routes_pdf['call_type'] = np.random.choice(['A', 'B', 'C'], size=len(routes_pdf), p=call_prob.values)
routes_pdf['taxi_id'] = np.random.choice(np.arange(1, CFG.taxi_pool_size + 1), size=len(routes_pdf)).astype(int)
routes_pdf[['timestamp', 'day_type', 'call_type', 'taxi_id']].head()

## 7) Write Dataset (Spark, Broadcast Join, Cache)

In [ ]:
final_pdf = routes_pdf[['trip_id', 'taxi_id', 'timestamp', 'origin_zone', 'destination_zone', 'polyline', 'call_type', 'day_type', 'distance_km', 'route_generated']].copy()
final_pdf = final_pdf.sort_values('trip_id')

sdf = spark.createDataFrame(final_pdf).cache()
_ = sdf.count()

zone_dim = spark.createDataFrame(zones_gdf[['zone_id', 'zone_name']])
sdf = (sdf
       .join(F.broadcast(zone_dim.withColumnRenamed('zone_id', 'origin_zone').withColumnRenamed('zone_name', 'origin_zone_name')), on='origin_zone', how='left')
       .join(F.broadcast(zone_dim.withColumnRenamed('zone_id', 'destination_zone').withColumnRenamed('zone_name', 'destination_zone_name')), on='destination_zone', how='left'))

output_path = CFG.output_s3
try:
    sdf.write.mode('overwrite').parquet(output_path)
    print('Saved to S3A:', output_path)
except Exception as e:
    print('S3A unavailable, fallback local:', e)
    output_path = CFG.output_local
    os.makedirs(output_path, exist_ok=True)
    sdf.write.mode('overwrite').parquet(output_path)
    print('Saved locally:', output_path)

output_path

## 8) Validation and Visualisation

In [ ]:
# Distance histogram: synthetic vs Porto
plt.figure(figsize=(10, 5))
bins = np.linspace(0, 40, 80)
pdist = porto_distances[(porto_distances > 0.2) & (porto_distances < 80)]
sdist = final_pdf['distance_km'][(final_pdf['distance_km'] > 0.2) & (final_pdf['distance_km'] < 80)]
plt.hist(pdist, bins=bins, alpha=0.55, density=True, label='Porto')
plt.hist(sdist, bins=bins, alpha=0.55, density=True, label='Synthetic Casablanca')
plt.xlabel('Distance (km)')
plt.ylabel('Density')
plt.title('Trip Distance Distribution Comparison')
plt.legend()
plt.show()

In [ ]:
# Folium heatmap of origins
import folium

center = [zones_gdf['centroid_lat'].mean(), zones_gdf['centroid_lon'].mean()]
m = folium.Map(location=center, zoom_start=11, tiles='CartoDB positron')
heat = routes_pdf[['origin_lat', 'origin_lon']].dropna().values.tolist()
HeatMap(heat, radius=7, blur=6, min_opacity=0.3).add_to(m)
m

In [ ]:
# Route overlay sample
fig, ax = ox.plot_graph(G_casa, node_size=0, edge_linewidth=0.4, edge_color='#bdbdbd', show=False, close=False, figsize=(10, 10), bgcolor='white')
sample_routes = final_pdf[final_pdf['route_generated']].sample(n=min(120, int(final_pdf['route_generated'].sum())), random_state=42)
for pl in sample_routes['polyline']:
    if not pl or len(pl) < 2:
        continue
    xs = [p[0] for p in pl]
    ys = [p[1] for p in pl]
    ax.plot(xs, ys, color='#e45756', alpha=0.35, linewidth=0.9)
ax.set_title('Sample Synthetic Routes - Casablanca')
plt.show()

In [ ]:
# 100% bbox check
lon_min, lon_max = zones_gdf['lon_min'].min(), zones_gdf['lon_max'].max()
lat_min, lat_max = zones_gdf['lat_min'].min(), zones_gdf['lat_max'].max()
ok_o = (routes_pdf['origin_lon'].between(lon_min, lon_max) & routes_pdf['origin_lat'].between(lat_min, lat_max)).mean()
ok_d = (routes_pdf['destination_lon'].between(lon_min, lon_max) & routes_pdf['destination_lat'].between(lat_min, lat_max)).mean()
print(f'Origins in Casablanca bbox: {ok_o*100:.2f}%')
print(f'Destinations in Casablanca bbox: {ok_d*100:.2f}%')

## Main Entry Point

In [ ]:
def run_pipeline(cfg: SimulationConfig):
    print('Notebook cells above execute the full pipeline top-to-bottom.')
    print('Final output path:', output_path)
    return output_path

if __name__ == '__main__':
    final_path = run_pipeline(CFG)
    print('Done:', final_path)